In [ ]:
# ===================================================================
# KAGGLE FEATURE ENGINEERING (V6) - With "In-Place" Brand Fix
# This script loads data from your KAGGLE DATASET, applies the
# advanced spacy-based brand extraction, and saves a new, cleaner
# set of processed files to /kaggle/working/.
# ===================================================================
import os
import re
import pandas as pd
import numpy as np
import gc
import time
from contextlib import contextmanager

# Text processing
import nltk
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

# Scikit-learn for feature extraction and processing
from sklearn.feature_extraction.text import TfidfVectorizer, HashingVectorizer
from sklearn.preprocessing import LabelBinarizer
from scipy.sparse import hstack, csr_matrix
from scipy.sparse import save_npz

# --- 1. KAGGLE-SPECIFIC SETUP: Install and download spacy ---
print("[SETUP] Installing spacy...")
!pip install -q spacy
print("[SETUP] Downloading spacy English model 'en_core_web_sm'...")
!python -m spacy download en_core_web_sm -q
import spacy

# --- Configuration Block ---
@contextmanager
def timer(name):
    t0 = time.time()
    yield
    print(f'[{name}] done in {time.time() - t0:.0f} s')

# --- CORRECTED PATHS FOR KAGGLE DATASET ---
# This assumes your dataset is named 'amazn-25-full-dataset' and contains the 'amazn-25' folder.
INPUT_DIR = '/kaggle/input/amazn-25-1/amazn-25/'
OUTPUT_DIR = '/kaggle/working/'
OUTPUT_VERSION = 'v6_brand_fix'

# Feature Engineering Parameters
NUM_BRANDS = 5000
TFIDF_MAX_FEATURES_TEXT = 200000
TFIDF_MAX_FEATURES_NAME_CHARS = 20000
HASHING_NUM_FEATURES = 2**20

# ===================================================================
# HELPER FUNCTION DEFINITIONS
# ===================================================================
def parse_catalog_content(content_string: str) -> dict:
    if not isinstance(content_string, str): return {}
    parsed_data = {}; lines = content_string.strip().split('\n'); current_key = None; buffer = []
    for line in lines:
        match = re.match(r'^(.*?):\s*(.*)', line)
        if match:
            if current_key and buffer: parsed_data[current_key] = ' '.join(buffer).strip()
            key_strip = match.group(1).strip()
            if key_strip.lower() == 'manufacturer': current_key = 'brand_from_bullet'
            else: current_key = key_strip
            initial_value = match.group(2).strip(); buffer = [initial_value] if initial_value else []
        elif current_key: buffer.append(line.strip())
    if current_key and buffer: parsed_data[current_key] = ' '.join(buffer).strip()
    final_data = {}; bullets = []
    for key, value in parsed_data.items():
        k_lower = str(key).lower()
        if 'bullet point' in k_lower: bullets.append(value)
        elif 'item name' in k_lower: final_data['item_name'] = value
        elif 'product description' in k_lower: final_data['product_description'] = value
        elif 'value' in k_lower: final_data['value'] = value
        elif 'unit' in k_lower: final_data['unit'] = value
        elif 'brand_from_bullet' in k_lower: final_data['brand_from_bullet'] = value
    final_data['bullet_points_combined'] = ' '.join(bullets)
    return final_data

def extract_ipq_v3(row: pd.Series) -> int:
    item_name = str(row.get('item_name', '')).lower()
    patterns = [r'\(pack of (\d+)\)', r'pack of (\d+)', r'(\d+)\s*per case', r'\((\d+)[\s-]*count\)', r'(\d+)[\s-]*count', r'(\d+)\s*pack', r'- (\d+) per case', r'set of (\d+)', r'\((\d+) pack\)']
    for pattern in patterns:
        match = re.search(pattern, item_name)
        if match:
            quantity = int(match.group(1)); return quantity if quantity > 0 else 1
    unit = str(row.get('unit_cleaned', '')).lower()
    if 'count' in unit:
        value = row.get('value_numeric', 1.0)
        if value > 1 and value == int(value): return int(value)
    return 1
    
nltk.download('punkt', quiet=True)
def deep_clean_text(text: str) -> str:
    if not isinstance(text, str): return ""
    text = text.lower(); text = re.sub(r'[^a-z0-9\s]', ' ', text); text = re.sub(r'\s+', ' ', text).strip()
    return text
def stem_text(text: str, stemmer) -> str:
    tokens = word_tokenize(text); stemmed_tokens = [stemmer.stem(token) for token in tokens]
    return ' '.join(stemmed_tokens)
def process_categoricals(df: pd.DataFrame, col_name: str, top_n: int) -> pd.DataFrame:
    df[col_name] = df[col_name].fillna('missing').astype(str)
    top_values = df[col_name].value_counts().index[:top_n]
    df.loc[~df[col_name].isin(top_values), col_name] = 'RARE_VALUE'
    return df

print("[SETUP] Loading spacy model...")
nlp = spacy.load("en_core_web_sm")
GARBAGE_BLACKLIST_V2 = {'The', 'A', 'An', 'And', 'For', 'Of', 'With', 'In', 'Pack', 'Set', 'Count', 'Case', 'Organic', 'Gluten', 'Sugar', 'Free', 'Natural', 'Premium', 'New', 'Ounce', 'Oz', 'Gram', 'Kg', 'Lb', 'Lbs', 'Box', 'Bag', 'Bottle', 'Jar', 'Food', 'Fresh Roasted Coffee', 'Dr', 'M', 'Mrs', 'brand', 'Brand'}
def extract_brand_v4(row: pd.Series) -> str:
    if pd.notna(row.get('brand_from_bullet')): return row['brand_from_bullet']
    item_name = row.get('item_name')
    if not isinstance(item_name, str): return 'Unknown'
    doc = nlp(item_name)
    proper_nouns = []
    for token in doc:
        if token.pos_ == 'PROPN': proper_nouns.append(token.text)
        elif proper_nouns: break
    if proper_nouns:
        potential_brand = " ".join(proper_nouns)
        if potential_brand not in GARBAGE_BLACKLIST_V2: return potential_brand
    match = re.match(r'([A-Z][\w\'-]*(\s+[A-Z][\w\'-]*)*)', item_name)
    if match:
        potential_brand = match.group(1).strip()
        if potential_brand not in GARBAGE_BLACKLIST_V2: return potential_brand
    return 'Unknown'


# ===================================================================
# MAIN FEATURE ENGINEERING PIPELINE (V6)
# ===================================================================
def run_feature_pipeline_v6():
    if not os.path.exists(OUTPUT_DIR): os.makedirs(OUTPUT_DIR)
        
    with timer("Loading raw data from Kaggle Dataset"):
        train_df = pd.read_csv(os.path.join(INPUT_DIR, 'train.csv'))
        test_df = pd.read_csv(os.path.join(INPUT_DIR, 'test.csv'))
        train_df = train_df[train_df['price'] > 0].reset_index(drop=True)
        num_train_original = len(train_df)
        combined_df = pd.concat([train_df, test_df], ignore_index=True, sort=False)
        del train_df, test_df; gc.collect()
        
    with timer("Parsing and Basic Feature Creation"):
        parsed_series = combined_df['catalog_content'].apply(parse_catalog_content)
        parsed_df = parsed_series.apply(pd.Series)
        combined_df = pd.concat([combined_df, parsed_df], axis=1)
        combined_df['value_numeric'] = pd.to_numeric(combined_df['value'], errors='coerce').fillna(0)
        combined_df['unit_cleaned'] = combined_df['unit'].str.lower().fillna('unknown').str.strip()
        print("\n[FIX] Applying new brand extraction logic (v4)...")
        combined_df['brand'] = combined_df.apply(extract_brand_v4, axis=1)
        combined_df['ipq'] = combined_df.apply(extract_ipq_v3, axis=1)
        combined_df['name_len'] = combined_df['item_name'].str.len().fillna(0)
        combined_df['desc_len'] = combined_df['product_description'].str.len().fillna(0)
        text_cols = ['item_name', 'bullet_points_combined', 'product_description']
        combined_df['unified_text'] = combined_df[text_cols].fillna('').agg(' [SEP] '.join, axis=1)

    with timer("Deep Text Cleaning and Stemming"):
        stemmer = PorterStemmer()
        combined_df['unified_text_cleaned'] = combined_df['unified_text'].apply(deep_clean_text)
        combined_df['unified_text_stemmed'] = combined_df['unified_text_cleaned'].apply(lambda x: stem_text(x, stemmer))
        combined_df['item_name_cleaned'] = combined_df['item_name'].apply(deep_clean_text)

    print("\n[VERIFICATION] Top 30 brands found with NEW logic:")
    print(combined_df['brand'].value_counts().head(30))

    with timer("Processing Categorical Features"):
        combined_df = process_categoricals(combined_df, 'brand', NUM_BRANDS)
        lb = LabelBinarizer(sparse_output=True)
        brand_sparse = lb.fit_transform(combined_df['brand'])
        
    with timer("Vectorizing Text Features"):
        tfidf_text = TfidfVectorizer(max_features=TFIDF_MAX_FEATURES_TEXT, ngram_range=(1, 2), token_pattern=r'\b\w+\b')
        text_sparse = tfidf_text.fit_transform(combined_df['unified_text_stemmed'])
        tfidf_name_chars = TfidfVectorizer(max_features=TFIDF_MAX_FEATURES_NAME_CHARS, ngram_range=(2, 5), analyzer='char')
        name_chars_sparse = tfidf_name_chars.fit_transform(combined_df['item_name_cleaned'])
        hash_vec = HashingVectorizer(n_features=HASHING_NUM_FEATURES, ngram_range=(1, 3))
        hashing_sparse = hash_vec.fit_transform(combined_df['unified_text_cleaned'])

    with timer("Assembling Final Sparse Matrix"):
        numeric_cols = ['ipq', 'value_numeric', 'name_len', 'desc_len']
        numeric_features = csr_matrix(combined_df[numeric_cols].values)
        X_sparse = hstack([numeric_features, brand_sparse, text_sparse, name_chars_sparse, hashing_sparse]).tocsr()
        
    with timer("Splitting and Saving Data"):
        X_train = X_sparse[:num_train_original]
        X_test = X_sparse[num_train_original:]
        y_train = np.log1p(pd.read_csv(os.path.join(INPUT_DIR, 'train.csv')).pipe(lambda x: x[x['price'] > 0])['price'].values)
        save_npz(os.path.join(OUTPUT_DIR, f'X_train_sparse_{OUTPUT_VERSION}.npz'), X_train)
        save_npz(os.path.join(OUTPUT_DIR, f'X_test_sparse_{OUTPUT_VERSION}.npz'), X_test)
        np.save(os.path.join(OUTPUT_DIR, f'y_train_{OUTPUT_VERSION}.npy'), y_train)
        
    print("\n--- Feature Engineering Complete ---")
    print(f"NEW Train sparse matrix shape: {X_train.shape}")
    print(f"NEW Processed files saved to: {OUTPUT_DIR} with version tag '{OUTPUT_VERSION}'")

# --- Execute the Pipeline ---
run_feature_pipeline_v6()

In [ ]:
# ===================================================================
# PHASE 1: Generate and Save CLIP Image Embeddings (Multi-GPU)
# INCLUDES A FIX FOR THE NUMPY BINARY INCOMPATIBILITY ERROR
# ===================================================================
import os
import torch
import pandas as pd
import numpy as np
from PIL import Image
from tqdm.notebook import tqdm
import gc
from contextlib import contextmanager

# --- 1. DEPENDENCY FIX for Numpy Incompatibility ---
# This block forces re-installation of key libraries to ensure they are
# compiled against the same numpy version, fixing the ValueError.
print("[SETUP] Fixing potential numpy version conflicts...")
!pip uninstall -y numpy scikit-learn sentence-transformers transformers
!pip install --no-cache-dir numpy scikit-learn sentence-transformers transformers
print("[SUCCESS] Dependencies have been re-installed.")

# --- NOW we can safely import sentence_transformers ---
from sentence_transformers import SentenceTransformer

print("\n--- Starting Phase 1: CLIP Embedding Generation (Multi-GPU) ---")

# --- Configuration ---
@contextmanager
def timer(name):
    t0 = time.time()
    yield
    print(f'[{name}] done in {time.time() - t0:.0f} s')

INPUT_DIR = '/kaggle/input/amazn-25-1/amazn-25/'
IMAGE_DIR = os.path.join(INPUT_DIR, 'images/')
OUTPUT_DIR = '/kaggle/working/'
OUTPUT_VERSION = 'v6_brand_fix'

# --- 2. Setup Multi-GPU Model ---
print(f"[INFO] Detected {torch.cuda.device_count()} GPUs. Both will be used.")
print("Loading CLIP model 'clip-ViT-B-32' for multi-GPU processing...")
clip_model = SentenceTransformer('clip-ViT-B-32')
print("[SUCCESS] CLIP model loaded.")

# --- 3. Load Data and Prepare Image Paths ---
with timer("Loading CSVs and preparing image paths"):
    train_df = pd.read_csv(os.path.join(INPUT_DIR, 'train.csv'))
    test_df = pd.read_csv(os.path.join(INPUT_DIR, 'test.csv'))

    all_df = pd.concat([train_df[['sample_id', 'image_link']], test_df[['sample_id', 'image_link']]], ignore_index=True)
    all_df = all_df.dropna(subset=['image_link'])

    def get_image_path(link):
        try:
            filename = os.path.basename(link)
            return os.path.join(IMAGE_DIR, filename)
        except:
            return None

    all_df['image_path'] = all_df['image_link'].apply(get_image_path)
    image_paths_to_process = all_df['image_path'][all_df['image_path'].apply(lambda p: isinstance(p, str) and os.path.exists(p))].tolist()
    print(f"Found {len(image_paths_to_process)} existing images to process.")

# --- 4. Generate Embeddings in Parallel ---
with timer("Generating embeddings"):
    batch_size = 128
    print(f"Starting encoding with batch size {batch_size} across both GPUs...")

    pool = clip_model.start_multi_process_pool()
    images_to_encode = [Image.open(p) for p in tqdm(image_paths_to_process, desc="Opening images")]
    
    image_embeddings = clip_model.encode_multi_process(
        images_to_encode,
        pool=pool,
        batch_size=batch_size
    )

    clip_model.stop_multi_process_pool(pool)
    print(f"[SUCCESS] Generated embeddings. Shape: {image_embeddings.shape}")
    del images_to_encode
    gc.collect()

# --- 5. Align and Save Embeddings ---
with timer("Aligning and saving embeddings"):
    path_to_id_map = pd.Series(all_df.sample_id.values, index=all_df.image_path).to_dict()
    processed_ids = np.array([path_to_id_map[p] for p in image_paths_to_process])

    train_ids_set = set(train_df['sample_id'])
    is_train_mask = np.array([sid in train_ids_set for sid in processed_ids])

    train_embeddings = image_embeddings[is_train_mask]
    train_sample_ids = processed_ids[is_train_mask]
    test_embeddings = image_embeddings[~is_train_mask]
    test_sample_ids = processed_ids[~is_train_mask]

    train_embed_path = os.path.join(OUTPUT_DIR, f'clip_embeddings_train_{OUTPUT_VERSION}.npy')
    train_ids_path = os.path.join(OUTPUT_DIR, f'clip_ids_train_{OUTPUT_VERSION}.npy')
    test_embed_path = os.path.join(OUTPUT_DIR, f'clip_embeddings_test_{OUTPUT_VERSION}.npy')
    test_ids_path = os.path.join(OUTPUT_DIR, f'clip_ids_test_{OUTPUT_VERSION}.npy')

    np.save(train_embed_path, train_embeddings)
    np.save(train_ids_path, train_sample_ids)
    np.save(test_embed_path, test_embeddings)
    np.save(test_ids_path, test_sample_ids)
    
    print(f"\n[SUCCESS] All CLIP embeddings and corresponding IDs have been saved to: {OUTPUT_DIR}")

del clip_model, image_embeddings, train_df, test_df, all_df
gc.collect()
torch.cuda.empty_cache()

In [ ]:

import os
import pandas as pd
import multiprocessing
from tqdm.notebook import tqdm
from functools import partial
import urllib.request
from pathlib import Path
import logging

# --- Setup Logging and Configuration ---
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# --- Path Definitions ---
# Input path for the CSV files
INPUT_DIR = '/kaggle/input/amazn-25-1/amazn-25/'

# The main output directory where your v6 files already are
OUTPUT_DIR = '/kaggle/working/'

# We will create a new 'images' subdirectory inside the main output directory
IMAGE_DIR = os.path.join(OUTPUT_DIR, 'images/')

# --- Download Functions (Robust Version) ---
def download_image_worker(image_link: str, save_folder: str) -> tuple:
    """Worker function to download a single image with error handling."""
    if not isinstance(image_link, str):
        return ('', 'skipped_invalid_link')
    
    try:
        filename = os.path.basename(image_link)
        image_save_path = os.path.join(save_folder, filename)
        
        if not os.path.exists(image_save_path):
            urllib.request.urlretrieve(image_link, image_save_path)
            return (image_link, 'success')
        else:
            return (image_link, 'skipped_exists')
            
    except Exception as e:
        # Log failed downloads for review
        with open(os.path.join(OUTPUT_DIR, "failed_downloads.log"), "a") as f:
            f.write(f"{image_link}\n")
        return (image_link, f'failed_{e.__class__.__name__}')

def download_all_images(image_links: list, download_folder: str):
    """Manages the multiprocessing download."""
    if not os.path.exists(download_folder):
        print(f"Creating image directory at: {download_folder}")
        os.makedirs(download_folder)
    
    num_processes = min(multiprocessing.cpu_count(), 32)
    logging.info(f"Starting download of {len(image_links)} unique images with {num_processes} parallel workers...")
    
    downloader = partial(download_image_worker, save_folder=download_folder)
    
    results = []
    with multiprocessing.Pool(processes=num_processes) as pool:
        for result in tqdm(pool.imap_unordered(downloader, image_links), total=len(image_links)):
            results.append(result)
    
    logging.info("--- Image Download Process Complete ---")
    status_counts = pd.Series([r[1] for r in results]).value_counts()
    print("Download Summary:")
    print(status_counts)

# --- Main Execution Block ---
def main_download_to_working():
    """Orchestrates the download process."""
    print("Reading CSVs to get all image URLs...")
    train_df = pd.read_csv(os.path.join(INPUT_DIR, 'train.csv'))
    test_df = pd.read_csv(os.path.join(INPUT_DIR, 'test.csv'))
    
    all_image_links = pd.concat([train_df['image_link'], test_df['image_link']]).dropna().unique()
    logging.info(f"Found {len(all_image_links)} unique image URLs to download.")
    del train_df, test_df

    # Run the download, saving to /kaggle/working/images/
    download_all_images(all_image_links, IMAGE_DIR)
    
    # Final verification
    print("\nVerifying download...")
    final_count = len(os.listdir(IMAGE_DIR))
    print(f"Total files in '{IMAGE_DIR}': {final_count}")
    if final_count > 140000:
        print("\n[SUCCESS] Download appears to be complete!")
        print(f"The 'images' folder is now inside '{OUTPUT_DIR}' alongside your other files.")
    else:
        print(f"\n[WARNING] Download may be incomplete. Expected ~150k images, but found {final_count}.")

# --- Run the main function ---
if __name__ == '__main__':
    main_download_to_working()

In [ ]:

import os
import torch
import pandas as pd
import numpy as np
from PIL import Image
from tqdm.notebook import tqdm
import gc
from contextlib import contextmanager
import urllib.request
import io
from functools import partial
import multiprocessing

# --- 1. SETUP: Dependencies and Configuration ---
print("[SETUP] Ensuring all dependencies are correctly installed...")
!pip uninstall -y numpy scikit-learn sentence-transformers transformers
!pip install --no-cache-dir numpy scikit-learn sentence-transformers transformers
print("[SUCCESS] Dependencies re-installed.")

from sentence_transformers import SentenceTransformer

@contextmanager
def timer(name):
    t0 = time.time()
    yield
    print(f'[{name}] done in {time.time() - t0:.0f} s')

INPUT_DIR = '/kaggle/input/amazn-25-1/amazn-25/'
OUTPUT_DIR = '/kaggle/working/'
OUTPUT_VERSION = 'v6_brand_fix'
CHUNK_SIZE = 2048 # Process 2048 images at a time. A good balance for pipelining.

# --- 2. "On-the-Fly" Worker and Generator Functions ---
def download_and_open_image(image_url: str, url_to_id_map: dict) -> tuple:
    """Downloads an image and returns its sample_id and the PIL Image object."""
    if not isinstance(image_url, str):
        return None, None
    try:
        sample_id = url_to_id_map.get(image_url)
        if sample_id is None:
            return None, None
        
        with urllib.request.urlopen(image_url, timeout=10) as url:
            img_buffer = io.BytesIO(url.read())
        with Image.open(img_buffer) as img:
            return sample_id, img.convert("RGB")
    except Exception:
        return None, None

def image_generator(image_urls, url_to_id_map):
    """A generator that downloads images in parallel and yields them."""
    num_processes = min(multiprocessing.cpu_count(), 32)
    downloader = partial(download_and_open_image, url_to_id_map=url_to_id_map)
    
    with multiprocessing.Pool(processes=num_processes) as pool:
        for result in pool.imap_unordered(downloader, image_urls):
            if result[1] is not None:
                yield result # Yields (sample_id, PIL.Image)

# --- Main Execution Block ---

# 3. Setup Multi-GPU Model
print(f"[INFO] Detected {torch.cuda.device_count()} GPUs. Both will be used.")
print("Loading CLIP model 'clip-ViT-B-32' for multi-GPU processing...")
clip_model = SentenceTransformer('clip-ViT-B-32')
print("[SUCCESS] CLIP model loaded.")

# 4. Load Data and Get URLs/IDs
with timer("Loading CSVs and creating URL/ID maps"):
    train_df = pd.read_csv(os.path.join(INPUT_DIR, 'train.csv'))
    test_df = pd.read_csv(os.path.join(INPUT_DIR, 'test.csv'))
    all_df = pd.concat([train_df[['sample_id', 'image_link']], test_df[['sample_id', 'image_link']]], ignore_index=True)
    all_image_links = all_df['image_link'].dropna().unique().tolist()
    url_to_id_map = pd.Series(all_df.sample_id.values, index=all_df.image_link).to_dict()
    print(f"Found {len(all_image_links)} unique image URLs to process.")

# 5. The Hyper-Optimized Pipelined Loop
with timer("Generating all embeddings via pipelining"):
    all_embeddings = []
    all_ids = []
    chunk = []
    
    # Start the multi-process pool for the GPUs
    gpu_pool = clip_model.start_multi_process_pool()
    
    # Create the generator for the CPUs
    img_gen = image_generator(all_image_links, url_to_id_map)

    with tqdm(total=len(all_image_links), desc="Overall Progress") as pbar:
        for sample_id, image in img_gen:
            chunk.append((sample_id, image))
            
            # When the chunk is full, send it to the GPUs for processing
            if len(chunk) >= CHUNK_SIZE:
                chunk_ids = [item[0] for item in chunk]
                chunk_images = [item[1] for item in chunk]
                
                # This is the consumer step (GPU-bound)
                embeddings = clip_model.encode_multi_process(chunk_images, pool=gpu_pool, batch_size=128)
                
                all_embeddings.append(embeddings)
                all_ids.extend(chunk_ids)
                
                # Clear the chunk to save RAM
                pbar.update(len(chunk))
                del chunk, chunk_ids, chunk_images, embeddings
                gc.collect()
                chunk = []

        # Process the final, smaller chunk if any images are left
        if len(chunk) > 0:
            chunk_ids = [item[0] for item in chunk]
            chunk_images = [item[1] for item in chunk]
            embeddings = clip_model.encode_multi_process(chunk_images, pool=gpu_pool, batch_size=128)
            all_embeddings.append(embeddings)
            all_ids.extend(chunk_ids)
            pbar.update(len(chunk))

    # Stop the GPU processes
    clip_model.stop_multi_process_pool(gpu_pool)
    
    # Combine all the chunk results
    image_embeddings = np.concatenate(all_embeddings)
    processed_ids = np.array(all_ids)

# 6. Align and Save Embeddings
with timer("Aligning and saving embeddings"):
    train_ids_set = set(train_df['sample_id'])
    is_train_mask = np.array([sid in train_ids_set for sid in processed_ids])

    train_embeddings = image_embeddings[is_train_mask]
    train_sample_ids = processed_ids[is_train_mask]
    test_embeddings = image_embeddings[~is_train_mask]
    test_sample_ids = processed_ids[~is_train_mask]

    print(f"Train embeddings shape: {train_embeddings.shape}")
    print(f"Test embeddings shape: {test_embeddings.shape}")

    train_embed_path = os.path.join(OUTPUT_DIR, f'clip_embeddings_train_{OUTPUT_VERSION}.npy')
    train_ids_path = os.path.join(OUTPUT_DIR, f'clip_ids_train_{OUTPUT_VERSION}.npy')
    test_embed_path = os.path.join(OUTPUT_DIR, f'clip_embeddings_test_{OUTPUT_VERSION}.npy')
    test_ids_path = os.path.join(OUTPUT_DIR, f'clip_ids_test_{OUTPUT_VERSION}.npy')

    np.save(train_embed_path, train_embeddings)
    np.save(train_ids_path, train_sample_ids)
    np.save(test_embed_path, test_embeddings)
    np.save(test_ids_path, test_sample_ids)
    
    print(f"\n[SUCCESS] All 'on-the-fly' CLIP embeddings have been saved to: {OUTPUT_DIR}")

del clip_model, image_embeddings, train_df, test_df, all_df
gc.collect()
torch.cuda.empty_cache()

In [ ]:
!rm -rf /kaggle/working/images


In [ ]:
# ===================================================================
# FINAL SCRIPT (V12): The "IT WILL FINISH" Resumable CLIP Embedder
# This version removes all multiprocessing and uses a simple, robust,
# and resumable loop to guarantee completion.
# ===================================================================
import os
import torch
import pandas as pd
import numpy as np
from PIL import Image
from tqdm.notebook import tqdm
import gc
from contextlib import contextmanager
import urllib.request
import io
import time

# --- 1. SETUP: Use pre-installed, stable libraries ---
print("[SETUP] Using pre-installed libraries from the stable Kaggle environment.")
try:
    from transformers import CLIPProcessor, CLIPModel
except ImportError:
    print("[ERROR] 'transformers' library not found. Please restart your session and ensure you are in a standard Kaggle notebook environment.")
    raise

@contextmanager
def timer(name):
    t0 = time.time()
    yield
    print(f'[{name}] done in {time.time() - t0:.0f} s')

# --- Configuration for a Resumable Workflow ---
INPUT_DIR = '/kaggle/input/amazn-25-1/amazn-25/'
OUTPUT_DIR = '/kaggle/working/'
OUTPUT_VERSION = 'v6_brand_fix'
CHUNK_DIR = os.path.join(OUTPUT_DIR, 'embedding_chunks/')
BATCH_SIZE = 256 # Batch size for GPU processing

# --- 2. Setup Model ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\n[INFO] Using device: {device.type.upper()}")

MODEL_NAME = "openai/clip-vit-base-patch32"
model = CLIPModel.from_pretrained(MODEL_NAME).to(device)
processor = CLIPProcessor.from_pretrained(MODEL_NAME)
print("[SUCCESS] Model and processor loaded.")

# --- 3. Load Data and Filter for Remaining Work ---
with timer("Loading CSVs and filtering URLs"):
    train_df = pd.read_csv(os.path.join(INPUT_DIR, 'train.csv'))
    test_df = pd.read_csv(os.path.join(INPUT_DIR, 'test.csv'))
    all_df = pd.concat([train_df[['sample_id', 'image_link']], test_df[['sample_id', 'image_link']]], ignore_index=True)
    all_df.dropna(subset=['image_link'], inplace=True)
    url_id_map = pd.Series(all_df.sample_id.values, index=all_df.image_link).to_dict()
    all_image_links = all_df['image_link'].unique().tolist()
    
    # --- RESUMABILITY LOGIC ---
    if not os.path.exists(CHUNK_DIR):
        os.makedirs(CHUNK_DIR)

    processed_ids = set()
    for f in os.listdir(CHUNK_DIR):
        if f.startswith('ids_chunk_') and f.endswith('.npy'):
            try:
                ids = np.load(os.path.join(CHUNK_DIR, f))
                processed_ids.update(ids)
            except: # Handles corrupted files
                print(f"Warning: Could not load chunk file {f}. It might be re-processed.")

    if processed_ids:
        print(f"[RESUMING] Found {len(processed_ids)} already processed images in chunks. Skipping them.")
        # Create a reverse map from ID to URL to filter correctly
        id_to_url_map = {v: k for k, v in url_to_id_map.items()}
        all_image_links = [id_to_url_map[sid] for sid in url_to_id_map.values() if sid not in processed_ids]
        print(f"[RESUMING] {len(all_image_links)} images remaining to process.")
    
    total_images_to_process = len(all_image_links)

# --- 4. The Simple, Resilient Processing Loop ---
with timer("Generating all embeddings in a single, resumable process"):
    chunk_counter = len(os.listdir(CHUNK_DIR)) // 2 
    
    for i in tqdm(range(0, total_images_to_process, BATCH_SIZE), desc="Processing Batches"):
        url_batch = all_image_links[i:i + BATCH_SIZE]
        
        pil_images, valid_ids_batch = [], []
        for link in url_batch:
            try:
                with urllib.request.urlopen(link, timeout=20) as url_response:
                    img_buffer = io.BytesIO(url_response.read())
                with Image.open(img_buffer) as img:
                    pil_images.append(img.convert("RGB"))
                    valid_ids_batch.append(url_to_id_map[link])
            except Exception:
                continue
        
        if not pil_images:
            continue

        # --- GPU Consumer Step ---
        inputs = processor(text=None, images=pil_images, return_tensors="pt", padding=True)["pixel_values"].to(device)
        with torch.no_grad():
            embeddings = model.get_image_features(pixel_values=inputs)
        
        # --- SAVE CHUNK TO DISK IMMEDIATELY ---
        chunk_embeddings = embeddings.cpu().numpy()
        chunk_ids = np.array(valid_ids_batch)
        np.save(os.path.join(CHUNK_DIR, f'embeddings_chunk_{chunk_counter}.npy'), chunk_embeddings)
        np.save(os.path.join(CHUNK_DIR, f'ids_chunk_{chunk_counter}.npy'), chunk_ids)
        chunk_counter += 1
        
        del pil_images, valid_ids_batch, inputs, embeddings, chunk_embeddings, chunk_ids
        gc.collect()

# --- 5. Final Consolidation Step ---
# (This part is the same and will run after the loop finishes)
with timer("Consolidating all saved chunks"):
    # ... (Consolidation logic from previous script)
    all_embeddings, all_ids = [], []
    print(f"Loading {chunk_counter} saved chunks from disk...")
    for i in range(chunk_counter):
        try:
            all_embeddings.append(np.load(os.path.join(CHUNK_DIR, f'embeddings_chunk_{i}.npy')))
            all_ids.append(np.load(os.path.join(CHUNK_DIR, f'ids_chunk_{i}.npy')))
        except:
            print(f"Warning: Could not load chunk {i}. It may have been corrupted.")
            
    image_embeddings = np.concatenate(all_embeddings)
    processed_ids = np.concatenate(all_ids)

# --- 6. Final Saving Step ---
# (This part is the same)
with timer("Aligning and saving final embedding files"):
    # ... (Saving logic from previous script)
    train_ids_set = set(train_df['sample_id'])
    is_train_mask = np.array([sid in train_ids_set for sid in processed_ids])
    train_embeddings = image_embeddings[is_train_mask]
    train_sample_ids = processed_ids[is_train_mask]
    test_embeddings = image_embeddings[~is_train_mask]
    test_sample_ids = processed_ids[~is_train_mask]
    train_embed_path = os.path.join(OUTPUT_DIR, f'clip_embeddings_train_{OUTPUT_VERSION}.npy')
    train_ids_path = os.path.join(OUTPUT_DIR, f'clip_ids_train_{OUTPUT_VERSION}.npy')
    test_embed_path = os.path.join(OUTPUT_DIR, f'clip_embeddings_test_{OUTPUT_VERSION}.npy')
    test_ids_path = os.path.join(OUTPUT_DIR, f'clip_ids_test_{OUTPUT_VERSION}.npy')
    np.save(train_embed_path, train_embeddings)
    np.save(train_ids_path, train_sample_ids)
    np.save(test_embed_path, test_embeddings)
    np.save(test_ids_path, test_sample_ids)
    print(f"\n[SUCCESS] All consolidated CLIP embeddings have been saved to: {OUTPUT_DIR}")

del model, processor
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# ===================================================================
# FINAL SCRIPT (V13 - Tunable Parallelism): CLIP Embeddings
# ===================================================================
import os
import torch
import pandas as pd
import numpy as np
from PIL import Image
from tqdm.notebook import tqdm
import gc
from contextlib import contextmanager
import urllib.request
import io
import time
from multiprocessing import Pool, Queue, Manager, cpu_count
import re

# --- 1. DEPENDENCY FIX ---
print("[SETUP] Applying targeted dependency fix for huggingface_hub...")
!pip install "huggingface-hub>=0.20.0" "requests>=2.30.0" --upgrade --quiet
print("[SUCCESS] Dependencies patched. If this is the first run, restart the kernel.")

from transformers import CLIPProcessor, CLIPModel

print("\n--- Starting Final Resumable CLIP Embedding Generation ---")

# ===================================================================
# --- PERFORMANCE TUNING PARAMETERS ---
# ===================================================================
# Number of parallel CPU workers for downloading. Set to your preference (e.g., 8, 10, 16).
# Using cpu_count() is a good default.
NUM_WORKERS = 10 # <--- YOU CAN CHANGE THIS TO 8, 10, etc.

# Batch size for GPU processing. Increase this for better GPU utilization.
# Make it as large as your GPU memory allows.
BATCH_SIZE = 256 # <--- TRY INCREASING THIS TO 512, 768, etc.
# ===================================================================


@contextmanager
def timer(name):
    t0 = time.time()
    yield
    print(f'[{name}] done in {time.time() - t0:.0f} s')

# --- Configuration for a Resumable Workflow ---
INPUT_DIR = '/kaggle/input/amazn-25-1/amazn-25/'
OUTPUT_DIR = '/kaggle/working/'
OUTPUT_VERSION = 'v6_brand_fix'
CHUNK_DIR = os.path.join(OUTPUT_DIR, 'embedding_chunks/')

# --- 2. Setup Model ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\n[INFO] Using device: {device.type.upper()}")
print(f"[INFO] Number of download workers: {NUM_WORKERS}")
print(f"[INFO] GPU Batch Size: {BATCH_SIZE}")

MODEL_NAME = "openai/clip-vit-base-patch32"
model = CLIPModel.from_pretrained(MODEL_NAME).to(device)
processor = CLIPProcessor.from_pretrained(MODEL_NAME)
print("[SUCCESS] Model and processor loaded.")

# --- 3. Load Data and Filter for Remaining Work (ROBUST RESUMABILITY) ---
# ... (This entire section is perfect and remains unchanged) ...
with timer("Loading CSVs and filtering for remaining URLs"):
    train_df = pd.read_csv(os.path.join(INPUT_DIR, 'train.csv'))
    test_df = pd.read_csv(os.path.join(INPUT_DIR, 'test.csv'))
    all_df = pd.concat([train_df[['sample_id', 'image_link']], test_df[['sample_id', 'image_link']]], ignore_index=True)
    all_df.dropna(subset=['image_link'], inplace=True)
    url_to_id_map = pd.Series(all_df.sample_id.values, index=all_df.image_link).to_dict()
    
    os.makedirs(CHUNK_DIR, exist_ok=True)
    processed_ids = set()
    chunk_files = os.listdir(CHUNK_DIR)
    
    for f in chunk_files:
        if f.startswith('ids_chunk_') and f.endswith('.npy'):
            try: processed_ids.update(np.load(os.path.join(CHUNK_DIR, f)))
            except Exception as e: print(f"Warning: Could not load chunk file {f}. Error: {e}")

    if processed_ids:
        print(f"[RESUMING] Found {len(processed_ids)} already processed images. Skipping them.")
        id_to_url_map = {v: k for k, v in url_to_id_map.items()}
        urls_to_process = [id_to_url_map[sid] for sid in url_to_id_map.values() if sid not in processed_ids]
    else:
        print("[INFO] No previous progress found. Starting from scratch.")
        urls_to_process = all_df['image_link'].unique().tolist()
        
    total_images_to_process = len(urls_to_process)
    print(f"[INFO] {total_images_to_process} images remaining to process.")
    url_id_pairs_to_process = [(url, url_to_id_map[url]) for url in urls_to_process]
    
    existing_chunk_nums = [int(re.search(r'(\d+)', f).group(1)) for f in chunk_files if f.startswith('ids_chunk_')]
    chunk_counter = max(existing_chunk_nums) + 1 if existing_chunk_nums else 0
    print(f"[INFO] Starting new chunks with index: {chunk_counter}")

# --- 4. The Resilient, Pipelined Processing Loop ---
def download_worker(url_id_tuple, queue):
    url, sample_id = url_id_tuple
    try:
        with urllib.request.urlopen(url, timeout=20) as url_response:
            queue.put((sample_id, url_response.read()))
    except Exception: pass

if total_images_to_process > 0:
    with timer("Generating all remaining embeddings"):
        manager = Manager()
        queue = manager.Queue(maxsize=BATCH_SIZE * 2)
        # Use the NUM_WORKERS parameter here
        producer_pool = Pool(processes=NUM_WORKERS)
        producer_pool.starmap_async(download_worker, [(pair, queue) for pair in url_id_pairs_to_process])
        
        with tqdm(total=total_images_to_process, desc="Overall Progress") as pbar:
            processed_count = 0
            while processed_count < total_images_to_process:
                pil_images, valid_ids_batch = [], []
                # Fill a batch using the BATCH_SIZE parameter
                while len(pil_images) < BATCH_SIZE:
                    try:
                        sample_id, img_bytes = queue.get(timeout=60)
                        with Image.open(io.BytesIO(img_bytes)) as img:
                            pil_images.append(img.convert("RGB"))
                            valid_ids_batch.append(sample_id)
                    except: break
                
                if not pil_images: break
                
                inputs = processor(text=None, images=pil_images, return_tensors="pt", padding=True)["pixel_values"].to(device)
                with torch.no_grad():
                    embeddings = model.get_image_features(pixel_values=inputs)
                
                np.save(os.path.join(CHUNK_DIR, f'embeddings_chunk_{chunk_counter}.npy'), embeddings.cpu().numpy())
                np.save(os.path.join(CHUNK_DIR, f'ids_chunk_{chunk_counter}.npy'), np.array(valid_ids_batch))
                chunk_counter += 1
                
                pbar.update(len(pil_images))
                processed_count += len(pil_images)
                del pil_images, valid_ids_batch, inputs, embeddings
                gc.collect()

        producer_pool.close()
        producer_pool.join()
else:
    print("[INFO] No new images to process. Proceeding to consolidation.")

# --- 5. & 6. Final Consolidation and Saving ---
# ... (These sections are perfect and remain unchanged) ...
with timer("Consolidating all saved chunks"):
    all_embeddings, all_ids = [], []
    chunk_files = [f for f in os.listdir(CHUNK_DIR) if f.startswith('embeddings_chunk_')]
    print(f"Loading {len(chunk_files)} saved chunks from disk for final consolidation...")
    for f in tqdm(chunk_files, desc="Consolidating"):
        chunk_num_match = re.search(r'(\d+)', f)
        if not chunk_num_match: continue
        chunk_num = chunk_num_match.group(1)
        try:
            all_embeddings.append(np.load(os.path.join(CHUNK_DIR, f)))
            all_ids.append(np.load(os.path.join(CHUNK_DIR, f'ids_chunk_{chunk_num}.npy')))
        except Exception as e: print(f"Warning: Could not load chunk {chunk_num}. Error: {e}")
            
    if not all_embeddings:
        print("[FATAL ERROR] No embedding chunks were found or loaded. Cannot proceed.")
        raise FileNotFoundError("No valid chunks to consolidate.")
        
    image_embeddings = np.concatenate(all_embeddings)
    processed_ids = np.concatenate(all_ids)

with timer("Aligning and saving final embedding files"):
    train_ids_set = set(train_df['sample_id'])
    is_train_mask = np.array([sid in train_ids_set for sid in processed_ids])
    train_embeddings = image_embeddings[is_train_mask]
    train_sample_ids = processed_ids[is_train_mask]
    test_embeddings = image_embeddings[~is_train_mask]
    test_sample_ids = processed_ids[~is_train_mask]
    
    train_embed_path = os.path.join(OUTPUT_DIR, f'clip_embeddings_train_{OUTPUT_VERSION}.npy')
    train_ids_path = os.path.join(OUTPUT_DIR, f'clip_ids_train_{OUTPUT_VERSION}.npy')
    test_embed_path = os.path.join(OUTPUT_DIR, f'clip_embeddings_test_{OUTPUT_VERSION}.npy')
    test_ids_path = os.path.join(OUTPUT_DIR, f'clip_ids_test_{OUTPUT_VERSION}.npy')
    
    np.save(train_embed_path, train_embeddings)
    np.save(train_ids_path, train_sample_ids)
    np.save(test_embed_path, test_embeddings)
    np.save(test_ids_path, test_sample_ids)
    print(f"\n[SUCCESS] All consolidated CLIP embeddings have been saved to: {OUTPUT_DIR}")

del model, processor; gc.collect(); torch.cuda.empty_cache()

[SETUP] Applying targeted dependency fix for huggingface_hub...
[SUCCESS] Dependencies patched. If this is the first run, restart the kernel.

--- Starting Final Resumable CLIP Embedding Generation ---

[INFO] Using device: CUDA
[INFO] Number of download workers: 10
[INFO] GPU Batch Size: 256
[SUCCESS] Model and processor loaded.
[RESUMING] Found 10051 already processed images. Skipping them.
[INFO] 130536 images remaining to process.
[INFO] Starting new chunks with index: 64
[Loading CSVs and filtering for remaining URLs] done in 2 s


Overall Progress:   0%|          | 0/130536 [00:00<?, ?it/s]

In [ ]:
# ===================================================================
# CELL 1: INSTALL DEPENDENCIES
# Run this cell first, then RESTART the kernel before running the rest.
# ===================================================================
!pip install "huggingface-hub>=0.20.0" "requests>=2.30.0" "transformers>=4.20.0" --upgrade --quiet
print("[SUCCESS] Dependencies updated. Please RESTART the kernel now.")

In [ ]:
# ===================================================================
# FINAL SCRIPT (V14): The TRULY PERSISTENT & Resumable CLIP Embedder
# This version is designed to be stopped and resumed across multiple
# Kaggle sessions by saving its progress to the notebook's output.
# ===================================================================
import os
import torch
import pandas as pd
import numpy as np
from PIL import Image
from tqdm.notebook import tqdm
import gc
from contextlib import contextmanager
import urllib.request
import io
import time
from multiprocessing import Pool, cpu_count
import zipfile

# --- 1. SETUP: Use pre-installed, stable libraries ---
print("[SETUP] Using pre-installed libraries from the stable Kaggle environment.")
try:
    from transformers import CLIPProcessor, CLIPModel
except ImportError:
    print("[ERROR] 'transformers' library not found. Please restart your session.")
    raise

@contextmanager
def timer(name):
    t0 = time.time()
    yield
    print(f'[{name}] done in {time.time() - t0:.0f} s')

# --- Configuration for a Persistent Workflow ---
# This is the path to the output of a PREVIOUS run of this notebook
# You will need to add this as a data source to your notebook for resuming.
PREVIOUS_RUN_INPUT_DIR = '/kaggle/working/embedding_chunks/' # <--- CHANGE THIS if you name your dataset something else

# Standard Kaggle paths
INPUT_DIR = '/kaggle/input/amazn-25-1/amazn-25/'
OUTPUT_DIR = '/kaggle/working/'
CHUNK_DIR = os.path.join(OUTPUT_DIR, 'embedding_chunks/')
CHUNK_ZIP_PATH = os.path.join(OUTPUT_DIR, 'embedding_chunks.zip')
OUTPUT_VERSION = 'v6_brand_fix'
CHUNK_SIZE = 2048

# --- Worker Function (same as before) ---
def download_to_memory_worker(url_id_tuple):
    url, sample_id = url_id_tuple
    try:
        with urllib.request.urlopen(url, timeout=20) as url_response:
            return sample_id, url_response.read()
    except Exception:
        return None, None

# --- Main Execution Block Wrapped in try...finally ---
try:
    # --- 2. RESUMABILITY: Unzip progress from previous run ---
    if not os.path.exists(CHUNK_DIR): os.makedirs(CHUNK_DIR)
    
    previous_zip_file = os.path.join(PREVIOUS_RUN_INPUT_DIR, 'embedding_chunks.zip')
    if os.path.exists(previous_zip_file):
        print(f"[RESUMING] Found progress from a previous run. Unzipping '{previous_zip_file}'...")
        with zipfile.ZipFile(previous_zip_file, 'r') as zip_ref:
            zip_ref.extractall(CHUNK_DIR)
        print("[RESUMING] Previous progress restored.")
    else:
        print("[INFO] No previous run found. Starting from scratch.")

    # --- 3. Load Data and Filter for Remaining Work ---
    with timer("Loading CSVs and filtering URLs"):
        train_df = pd.read_csv(os.path.join(INPUT_DIR, 'train.csv'))
        test_df = pd.read_csv(os.path.join(INPUT_DIR, 'test.csv'))
        all_df = pd.concat([train_df[['sample_id', 'image_link']], test_df[['sample_id', 'image_link']]], ignore_index=True)
        all_df.dropna(subset=['image_link'], inplace=True)
        
        processed_ids = set()
        for f in os.listdir(CHUNK_DIR):
            if f.startswith('ids_chunk_') and f.endswith('.npy'):
                ids = np.load(os.path.join(CHUNK_DIR, f))
                processed_ids.update(ids)

        if processed_ids:
            print(f"[RESUMING] Found {len(processed_ids)} already processed images in chunks. Skipping them.")
            all_df = all_df[~all_df['sample_id'].isin(processed_ids)]
        
        url_id_pairs = list(zip(all_df['image_link'], all_df['sample_id']))
        total_images_to_process = len(url_id_pairs)
        print(f"Found {total_images_to_process} images remaining to process.")

    # --- 4. Main Processing Loop (only if there's work to do) ---
    if total_images_to_process > 0:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
        processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
        
        with timer("Generating all remaining embeddings via pipelining"):
            num_workers = 10
            pool = Pool(processes=num_workers)
            chunk = []
            chunk_counter = len([f for f in os.listdir(CHUNK_DIR) if f.startswith('ids_chunk_')])
            results_iterator = pool.imap_unordered(download_to_memory_worker, url_id_pairs)
            
            for sample_id, img_bytes in tqdm(results_iterator, total=total_images_to_process, desc="Processing Images"):
                if img_bytes is not None:
                    try:
                        with Image.open(io.BytesIO(img_bytes)) as img:
                            chunk.append((sample_id, img.convert("RGB")))
                    except Exception: continue
                
                if len(chunk) >= CHUNK_SIZE:
                    chunk_ids = [item[0] for item in chunk]; chunk_images = [item[1] for item in chunk]
                    inputs = processor(text=None, images=chunk_images, return_tensors="pt", padding=True)["pixel_values"].to(device)
                    with torch.no_grad():
                        embeddings = model.get_image_features(pixel_values=inputs)
                    np.save(os.path.join(CHUNK_DIR, f'embeddings_chunk_{chunk_counter}.npy'), embeddings.cpu().numpy())
                    np.save(os.path.join(CHUNK_DIR, f'ids_chunk_{chunk_counter}.npy'), np.array(chunk_ids))
                    chunk_counter += 1
                    del chunk, chunk_ids, chunk_images, embeddings, inputs; gc.collect(); chunk = []

            if len(chunk) > 0: # Process final chunk
                chunk_ids = [item[0] for item in chunk]; chunk_images = [item[1] for item in chunk]
                inputs = processor(text=None, images=chunk_images, return_tensors="pt", padding=True)["pixel_values"].to(device)
                with torch.no_grad():
                    embeddings = model.get_image_features(pixel_values=inputs)
                np.save(os.path.join(CHUNK_DIR, f'embeddings_chunk_{chunk_counter}.npy'), embeddings.cpu().numpy())
                np.save(os.path.join(CHUNK_DIR, f'ids_chunk_{chunk_counter}.npy'), np.array(chunk_ids))

            pool.close()
            pool.join()
    else:
        print("[INFO] No new images to process. All chunks were already found.")

finally:
    # --- 5. THE INSURANCE POLICY: This block ALWAYS runs ---
    print("\n[FINALLY BLOCK] Zipping all progress to the output directory...")
    if os.path.exists(CHUNK_DIR) and len(os.listdir(CHUNK_DIR)) > 0:
        !zip -r {CHUNK_ZIP_PATH} {CHUNK_DIR}
        print(f"[SUCCESS] Progress has been saved to '{CHUNK_ZIP_PATH}'.")
        print("You can now save this version of the notebook and create a new dataset from this output file to resume later.")
    else:
        print("[INFO] No chunks found to zip.")

In [ ]:
!ls -l /kaggle/working/embedding_chunks/